# Memory X-Ray v0.2 (amendment 1): S-single feasibility rerun

This reruns the S-single feasibility test (16 evidence subsets × 3 fixed orderings) with the two amendment-1 fixes:

- weights stored in float16, **all arithmetic in float32** (fixes the run-1 I3 failure);
- model pinned to commit `f39ac1d28e925b323eae81227eaba4464caced4e`, verified after loading.

The scenario and every threshold are unchanged. The full experiment is **not** started.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Then run every cell in order.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code
Upload the **new** `memory_xray_v02.zip`. Any older copy of the folder in this runtime is moved aside, not deleted.

In [ ]:
import os, time, zipfile
from google.colab import files
if os.path.isdir("memory_xray_v02"):
    os.rename("memory_xray_v02", f"memory_xray_v02_old_{int(time.time())}")
uploaded = files.upload()  # choose the new memory_xray_v02.zip
zipfile.ZipFile(next(iter(uploaded))).extractall(".")
%cd memory_xray_v02
import mxr
assert mxr.__version__ in ("0.2.1", "0.2.2", "0.2.3", "0.2.4"), f"stale package: {mxr.__version__}"
print("package", mxr.__version__)

## 2. Install dependencies and run the offline tests
These use tiny models built in memory, download nothing, and should all pass.

In [ ]:
!pip install -q "transformers>=4.41.2" "accelerate>=0.31" pytest
!python -m pytest -q

## 3. Run the corrected feasibility test
The run aborts before loading if the pinned commit can't be loaded and verified, and before scoring if the tokenization checks fail. Float32 arithmetic makes it slower than run 1, but it should still finish in a few minutes on a T4.

In [ ]:
!python run_feasibility.py --model microsoft/Phi-3-mini-4k-instruct --revision f39ac1d28e925b323eae81227eaba4464caced4e --storage-dtype float16 --compute-dtype float32

## 4. View the report

In [ ]:
import glob
from IPython.display import Markdown, display
runs = sorted(d for d in glob.glob("runs/s_single_*") if os.path.isdir(d) and d != "runs/s_single_20261017T180509Z")
run_dir = runs[-1]
print(run_dir)
display(Markdown(open(os.path.join(run_dir, "REPORT.md")).read()))

## 5. Download the run folder
Send this zip back for review.

In [ ]:
import shutil
archive = shutil.make_archive(run_dir, "zip", run_dir)
files.download(archive)